In [1]:
# مرحله 1: کلون کردن مخزن
!git clone https://github.com/mohammadnabia/DA_nnUNet.git

# مرحله 2: وارد فولدر پروژه
%cd DA_nnUNet

# مرحله 3: نصب پروژه در حالت editable
!pip install -e .


Cloning into 'DA_nnUNet'...
remote: Enumerating objects: 303, done.
remote: Counting objects: 100% (303/303), done.
remote: Compressing objects: 100% (276/276), done.
remote: Total 303 (delta 45), reused 266 (delta 23), pack-reused 0 (from 0)
Receiving objects: 100% (303/303), 2.58 MiB | 25.65 MiB/s, done.
Resolving deltas: 100% (45/45), done.
/content/DA_nnUNet
Obtaining file:///content/DA_nnUNet
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 7.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 MB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━

In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
!unzip -q "/content/drive/MyDrive/BraTS-peds2023/BraTS-PEDs-2023.zip" -d /content/BraTS_PEDs_2023


In [3]:
import os
import shutil
import json
import glob

source_dir = "/content/BraTS_PEDs_2023/ASNR-MICCAI-BraTS2023-PED-Challenge-TrainingData"
target_dir = "/content/imagesTs"
os.makedirs(target_dir, exist_ok=True)

with open('/content/drive/MyDrive/100_epoch_Deepest_training_on_bratspeds_finetune/splits_final.json', 'r') as f:
    splits = json.load(f)

val_cases = splits[0]['val']  # Fold 0

for case in val_cases:
    case_path = os.path.join(source_dir, case)
    if not os.path.isdir(case_path):
        continue
    try:
        shutil.copyfile(glob.glob(os.path.join(case_path, "*t1n.nii.gz"))[0], os.path.join(target_dir, f"{case}_0000.nii.gz"))
        shutil.copyfile(glob.glob(os.path.join(case_path, "*t1c.nii.gz"))[0], os.path.join(target_dir, f"{case}_0001.nii.gz"))
        shutil.copyfile(glob.glob(os.path.join(case_path, "*t2w.nii.gz"))[0], os.path.join(target_dir, f"{case}_0002.nii.gz"))
        shutil.copyfile(glob.glob(os.path.join(case_path, "*t2f.nii.gz"))[0], os.path.join(target_dir, f"{case}_0003.nii.gz"))
    except IndexError:
        print(f"⚠️ Missing modalities for case: {case}")

print("✅ Validation fold 0 cases copied to imagesTs.")


✅ Validation fold 0 cases copied to imagesTs.


In [9]:
!mkdir -p /content/nnUNet_results_baseline/Dataset139_BraTS2023/nnUNetTrainer_TL_FTen_Custom_100epochs__nnUNetPlans__3d_fullres/fold_0

# کپی checkpoint
!cp "/content/drive/MyDrive/100_epoch_Deepest_training_on_bratspeds_finetune/fold_0_epoch_100.pth" \
    /content/nnUNet_results_baseline/Dataset139_BraTS2023/nnUNetTrainer_TL_FTen_Custom_100epochs__nnUNetPlans__3d_fullres/fold_0/checkpoint_final.pth

# کپی plans.json
!cp "/content/drive/MyDrive/100_epoch_Deepest_training_on_bratspeds_finetune/nnUNetPlans.json" \
    /content/nnUNet_results_baseline/Dataset139_BraTS2023/nnUNetTrainer_TL_FTen_Custom_100epochs__nnUNetPlans__3d_fullres/plans.json

# کپی dataset.json
!cp "/content/drive/MyDrive/100_epoch_Deepest_training_on_bratspeds_finetune/dataset.json" \
    /content/nnUNet_results_baseline/Dataset139_BraTS2023/nnUNetTrainer_TL_FTen_Custom_100epochs__nnUNetPlans__3d_fullres/

# کپی splits_final.json
!cp "/content/drive/MyDrive/100_epoch_Deepest_training_on_bratspeds_finetune/splits_final.json" \
    /content/nnUNet_results_baseline/Dataset139_BraTS2023/nnUNetTrainer_TL_FTen_Custom_100epochs__nnUNetPlans__3d_fullres/


In [5]:
import os

os.environ['nnUNet_raw'] = "/content/nnUNet_raw_baseline"
os.environ['nnUNet_preprocessed'] = "/content/nnUNet_preprocessed_baseline"
os.environ['nnUNet_results'] = "/content/nnUNet_results_baseline"


In [6]:
!mkdir -p /content/nnUNet_raw_baseline
!mkdir -p /content/nnUNet_preprocessed_baseline
!mkdir -p /content/nnUNet_results_baseline


In [7]:
import re

file_path = '/content/DA_nnUNet/nnunetv2/inference/predict_from_raw_data.py'
with open(file_path, 'r') as f:
    code = f.read()

# تغییر خط prediction
code = re.sub(r'prediction, _ = self\.network\(x\)', 'prediction = self.network(x)', code)

with open(file_path, 'w') as f:
    f.write(code)

print("✅ خط network(x) اصلاح شد!")


✅ خط network(x) اصلاح شد!


قبل از احرای این بخش
تابع زیر رو حتما به TRAINER  اضافه کن


```
import os
import shutil
import torch
from nnunetv2.training.nnUNetTrainer.nnUNetTrainer import nnUNetTrainer

class nnUNetTrainer_TL_FTen_FullFineTune_100epochs(nnUNetTrainer):
    """
    Trainer با:
    - آموزش کامل تمام لایه‌ها (full fine-tuning)
    - 100 ایپاک
    - ذخیره split
    - ذخیره checkpoint هر 10 ایپاک
    """
    def __init__(self, plans, configuration, fold, dataset_json, unpack_dataset=True, device='cuda'):
        super().__init__(plans, configuration, fold, dataset_json, unpack_dataset, device)
        self.num_epochs = 100  # آموزش به مدت 100 ایپاک

    def load_dataset(self):
        super().load_dataset()
        split_src = os.path.join(self.dataset_dir, "splits_final.json")
        split_dst = "/content/drive/MyDrive/nnUNet_checkpoints/splits_final.json"
        if os.path.exists(split_src):
            shutil.copy(split_src, split_dst)
            self.print_to_log_file(f"✅ فایل split ذخیره شد در: {split_dst}", also_print_to_console=True)

    def initialize(self):
        super().initialize()
        # همه پارامترها آزاد هستند
        for name, param in self.network.named_parameters():
            param.requires_grad = True  # بدون فریز

    def on_epoch_end(self):
        super().on_epoch_end()
        epoch = self.current_epoch if hasattr(self, "current_epoch") else self.epoch
        if (epoch + 1) % 10 == 0 or (epoch + 1) == self.num_epochs:
            save_dir = "/content/drive/MyDrive/nnUNet_checkpoints"
            os.makedirs(save_dir, exist_ok=True)
            save_path = os.path.join(save_dir, f"fold_{self.fold}_epoch_{epoch + 1}.pth")
            print(f"🔔 ذخیره checkpoint در epoch {epoch + 1} ...")
            self.save_checkpoint(save_path)
            print(f"✅ مدل در {save_path} ذخیره شد.")

    def predict_sliding_window_return_logits(self, data):
        outputs = self.network(data)
        return outputs[0] if isinstance(outputs, tuple) else outputs

        
```


!nnUNetv2_train 139 3d_fullres 0 -tr nnUNetTrainer_TL_FTen_FullFineTune_100epochs


In [11]:
!nnUNetv2_predict \
  -i /content/imagesTs \
  -o /content/pred_fold0_noTTA \
  -d 139 \
  -c 3d_fullres \
  -f 0 \
  -tr nnUNetTrainer_TL_FTen_FullFineTune_100epochs \
  --disable_tta


#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
#######################################################################

There are 20 cases in the source folder
I am process 0 out of 1 (max process ID is 0, we start counting with 0!)
There are 20 cases that I would like to predict

Predicting BraTS-PED-00008-000:
perform_everything_on_device: True
100% 8/8 [00:02<00:00,  3.16it/s]
sending off prediction to background worker for resampling and export
done with BraTS-PED-00008-000

Predicting BraTS-PED-00021-000:
perform_everything_on_device: True
100% 8/8 [00:00<00:00, 18.83it/s]
sending off prediction to background worker for resampling and export
done with BraTS-PED-00021-000

Predicting BraTS-PED-00026-000:

In [12]:
!mkdir -p /content/drive/MyDrive/nnUNet_Pruned_Results_peds2023/BraTS-PEDS_DeepestFT100epoch_80pruned_noTTA
!cp -r /content/pred_fold0_noTTA/* /content/drive/MyDrive/nnUNet_Pruned_Results_peds2023/BraTS-PEDS_DeepestFT100epoch_80pruned_noTTA/
